<a href="https://colab.research.google.com/github/acg-team/Bioinfo4B/blob/main/Project/single_cell_differential_gene_expression/sc_DGE.ipynb" target="_blank">
  <img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Run this notebook in Google Colab">
</a>

In [ ]:
# run this cell to install dependencies in google colab
# System dependencies
!apt-get update -qq
!apt-get install -y r-base

# Python dependencies
!pip install scanpy pertpy anndata2ri rpy2

The next cell prints the versions from the active runtime and can confirm that the notebook is running in the expected environment.

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import logging
import random
from pathlib import Path
import importlib.metadata as importlib_metadata

import anndata2ri
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import pertpy
import rpy2.rinterface_lib.callbacks
import scanpy as sc
import seaborn as sns
from matplotlib.lines import Line2D
from rpy2 import robjects as ro
from rpy2.robjects import pandas2ri
from rpy2.robjects.conversion import localconverter

sc.settings.verbosity = 0
sc.settings.set_figure_params(dpi=100, frameon=False)
rpy2.rinterface_lib.callbacks.logger.setLevel(logging.ERROR)

random.seed(0)
np.random.seed(0)

pandas2ri.activate()
anndata2ri.activate()

%load_ext rpy2.ipython


In [ ]:
python_versions = {
    "scanpy": importlib_metadata.version("scanpy"),
    "pertpy": importlib_metadata.version("pertpy"),
    "rpy2": importlib_metadata.version("rpy2"),
}

print("Python package versions")
for name, version in python_versions.items():
    print(f"  {name}: {version}")


In [ ]:
%%R
suppressPackageStartupMessages({
    library(edgeR)
    library(SingleCellExperiment)
})

cat("R package versions\n")
cat("  edgeR: ", as.character(packageVersion("edgeR")), "\n", sep = "")
cat("  SingleCellExperiment: ", as.character(packageVersion("SingleCellExperiment")), "\n", sep = "")


## Load and inspect the dataset

We keep the original `pertpy.data.gehring_2019()` loading step so the notebook stays close to the teaching material and can be re-run without shipping a large input file in the repository.


In [ ]:
mdata = pertpy.data.gehring_2019()
mdata

In [ ]:
mdata.obs.head()

In [ ]:
print(f"Cells: {mdata.n_obs:,}")
print(f"Genes: {mdata.n_vars:,}")
print(f"Expression matrix dtype: {mdata.X.dtype}")
print("\nBatch distribution")
display(mdata.obs["batch"].value_counts().sort_index())
print("\nDose distribution (top 10)")
display(mdata.obs["dose_value"].astype(str).value_counts().head(10))
print("\nPerturbation distribution (top 10)")
display(mdata.obs["perturbation"].astype(str).value_counts().head(10))


## Explain and clean metadata

The full object contains many descriptive columns, but for this notebook we keep only the metadata needed for the main question:

- `perturbation`: defines the biological comparison of interest (`control` vs `BMP4`),
- `batch`: captures technical or experimental batch structure,
- `dose_value`: keeps track of dosage differences within the perturbation experiment.

Keeping a minimal `.obs` table makes the downstream model easier to explain while still preserving the variables needed for design and interpretation.


In [ ]:
mdata.obs = mdata.obs[["perturbation", "batch", "dose_value"]].copy()
mdata.obs.head()

Before testing differential expression, we confirm that `.X` contains raw counts and preserve those counts explicitly in a dedicated layer.


In [ ]:
np.max(mdata.X)

In [ ]:
mdata.layers["counts"] = mdata.X.copy()
mdata.obs["batch"] = mdata.obs["batch"].astype("category")
mdata.obs["dose_value"] = mdata.obs["dose_value"].astype("category")
mdata.obs["perturbation"] = mdata.obs["perturbation"].astype(str)

mdata


In [ ]:
print("Unique batches:", mdata.obs["batch"].cat.categories.tolist())
print("Number of dose levels:", len(mdata.obs["dose_value"].cat.categories))
print("Example dose levels:", list(mdata.obs["dose_value"].cat.categories[:10]))


## Basic quality control

We apply a light first-pass QC with the same thresholds often used in introductory tutorials:

- `min_genes=200`: removes extremely low-complexity barcodes that are unlikely to be intact cells,
- `min_cells=3`: removes genes detected in too few cells to support stable downstream summaries.

These are teaching defaults rather than universal rules; in a real analysis you would revisit them after inspecting the dataset.


In [ ]:
sc.pp.calculate_qc_metrics(mdata, inplace=True)


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

sns.histplot(mdata.obs["n_genes_by_counts"], bins=40, ax=axes[0], color="#4C72B0")
axes[0].axvline(200, color="black", linestyle="--", linewidth=1)
axes[0].set_title("Genes detected per cell")
axes[0].set_xlabel("n_genes_by_counts")

sns.histplot(mdata.obs["total_counts"], bins=40, ax=axes[1], color="#55A868")
axes[1].set_title("Counts per cell")
axes[1].set_xlabel("total_counts")

batch_counts = mdata.obs["batch"].astype(str).value_counts().sort_index()
sns.barplot(x=batch_counts.index, y=batch_counts.values, ax=axes[2], color="#C44E52")
axes[2].set_title("Batch composition")
axes[2].set_xlabel("batch")
axes[2].set_ylabel("cells")

plt.tight_layout()


In [ ]:
sc.pp.filter_cells(mdata, min_genes=200)
sc.pp.filter_genes(mdata, min_cells=3)
mdata


## Exploratory analysis and clustering

Before fitting a differential expression model, it helps to look at the global structure of the dataset. Clustering is useful here because it tells us whether cells group by major biological response, by batch, or by some mixture of both.

For exploratory structure we use normalized and log-transformed data, while the DE test later returns to raw counts through pseudobulk aggregation.


In [ ]:
adata_clust = mdata.copy()
adata_clust.X = adata_clust.layers["counts"].copy()

sc.pp.normalize_total(adata_clust, target_sum=1e4)
sc.pp.log1p(adata_clust)
sc.pp.highly_variable_genes(adata_clust, n_top_genes=2000, subset=True)
sc.pp.scale(adata_clust, max_value=10)
sc.tl.pca(adata_clust, svd_solver="arpack")
sc.pp.neighbors(adata_clust, n_neighbors=15, n_pcs=30)
sc.tl.umap(adata_clust)
sc.tl.leiden(adata_clust, resolution=0.5)


In [ ]:
sc.pl.pca(adata_clust, color=["batch", "dose_value"], ncols=2)
sc.pl.umap(adata_clust, color=["leiden", "batch", "dose_value"], ncols=3)


**Interpretation note:** in a perturbation experiment like this one, we expect part of the structure to reflect treatment and dose, while residual separation by batch would suggest a technical effect that needs to be modeled explicitly. Compare the UMAP qualitatively to the article and ask whether the dominant axes seem biological, technical, or mixed.


## Define the BMP4 comparison

The full dataset contains many perturbations. For the teaching example below, we isolate a simple contrast between `control` and `BMP4`, while still accounting for `batch` and `dose_value`.

The exact spelling of the control label can differ between datasets, so the next cell lists the available perturbation categories first.


In [ ]:
perturbation_counts = mdata.obs["perturbation"].value_counts().sort_values(ascending=False)
perturbation_counts.head(20)


In [ ]:
## Your code here

## Build pseudobulk groups

Pseudobulk aggregation reduces the single-cell counts to sample-level profiles. Here we combine cells that share both batch and dose so the aggregation respects the structure of the experiment.


In [ ]:
## Your code here

## Differential expression design

We use an `edgeR` generalized linear model on pseudobulk counts. The design matrix includes:

- `batch`: to absorb technical or experimental batch effects,
- `dose_value`: to account for dosage differences,
- `condition`: the main biological effect of interest (`BMP4` vs `control`).

This is a compact and interpretable design for teaching. The key coefficient is the `conditionBMP4` term.


In [ ]:
## Your code here

## Present the results

We start with the strongest hits from the `edgeR` table, then visualize the global DE pattern using a volcano plot and a heatmap of the top genes.


In [ ]:
## Your code here

### Biological interpretation

After running the notebook, inspect the top genes and ask:

- which genes increase most strongly under BMP4,
- which genes decrease most strongly,
- whether those shifts are consistent with a BMP signaling response and altered neural stem cell state,
- whether batch adjustment changes the ranking compared with a naive unadjusted comparison.

A good discussion point is that strong-looking single-cell differences can shrink or disappear after pseudobulk aggregation and batch-aware modeling.


In [ ]:
## Your code here

## Questions

1. How many batches and dose levels are in the dataset?
2. What is the effect of batch on the pseudobulk groups?
3. Which gene has the strongest BMP4 response after FDR correction?
4. Do the UMAP and pseudobulk DE results tell a consistent story?
5. What changes if you remove `dose_value` from the design matrix?


## References

- sc-best-practices differential expression chapter: https://www.sc-best-practices.org/conditions/differential_gene_expression.html
- Gehring et al. 2019: https://www.nature.com/articles/s41587-019-0372-z
